# Sistemas Intensisvos en Datos - 202620

## Proyecto del Curso - 1ra Entrega

In [0]:
!pip install pathlib

## Índice
- [1. Contexto](#0-importación-de-librerías)
- [2. Ingesta de datos](#2-ingesta-de-datos)
- [3. Preguntas de negocio](#3-preguntas-de-negocio)

## 0. Importación de librerías

Se hace la importación de las librerias necesarias para la ejecución del _notebook_ relacionado con el proyecto de **OceanWatch Analytics**.

In [0]:
from pathlib import Path
import requests
import time
import zipfile


## 1. Ingesta de datos

In [0]:
# --- Configuración ---

# Creación del volumen
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.proyecto")

# URL para la descarga de los archivos en NOAA
base_url = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/"

# Archivos a descargar
files = [f"AIS_2023_06_{day:02d}.zip" for day in range(1, 8)]  # 01 a 07

# Directorios

# Directorio del volumen principal en Databricks
base_path = Path("/Volumes/workspace/default/proyecto/")

# Directorio para los archivos descargados
raw_dir = base_path / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Directorio para los archivos extraídos
dest = base_path / "extraido"
dest.mkdir(exist_ok=True)

# Configuración de descarga
MAX_RETRIES = 4
TIMEOUT = 60  # segundos por intento
CHUNK_SIZE = 1024 * 1024  # 1 MB


Definición de constantes

In [0]:
def verify_zip_integrity(path):
    """Verifica que el zip no esté corrupto revisando el CRC de cada archivo interno."""
    try:
        with zipfile.ZipFile(path) as z:
            bad_file = z.testzip()  # None si todo está bien; si no, devuelve el nombre del primer archivo corrupto
            if bad_file is not None:
                raise IOError(f"Archivo corrupto dentro del zip: {bad_file}")
            # Chequeo extra: que efectivamente tenga contenido (al menos 1 CSV)
            names = z.namelist()
            if not names:
                raise IOError("El zip está vacío")
            print(f"✓ Integridad OK: {path.name} ({len(names)} archivo(s) interno(s), CRC válido)")
            return True
    except zipfile.BadZipFile:
        raise IOError(f"{path.name} no es un zip válido (posiblemente descarga incompleta o corrupta)")

In [0]:

def download_with_retries(filename, max_retries=MAX_RETRIES):
    url = base_url + filename
    dest = raw_dir / filename

    # Si ya existe y no está vacío/corrupto, lo saltamos (idempotencia)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"✓ {filename} ya existe ({dest.stat().st_size / 1e6:.1f} MB), se omite descarga")
        return dest

    for attempt in range(1, max_retries + 1):
        try:
            print(f"↓ Descargando {filename} (intento {attempt}/{max_retries})...")
            with requests.get(url, stream=True, timeout=TIMEOUT) as r:
                r.raise_for_status()
                expected_size = int(r.headers.get("Content-Length", 0))

                tmp_dest = dest.with_suffix(".part")
                with open(tmp_dest, "wb") as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)

                actual_size = tmp_dest.stat().st_size

                # --- Verificación de integridad: tamaño esperado vs recibido ---
                if expected_size and actual_size != expected_size:
                    raise IOError(
                        f"Tamaño incompleto: esperado {expected_size} bytes, "
                        f"recibido {actual_size} bytes"
                    )

                tmp_dest.rename(dest)

                # --- Verificación de integridad (nivel 2: CRC interno del zip) ---
                verify_zip_integrity(dest)

                print(f"✓ {filename} descargado correctamente ({actual_size / 1e6:.1f} MB)")
                return dest

        except (requests.RequestException, IOError) as e:
            print(f"  ✗ Falló intento {attempt}: {e}")
            if tmp_dest.exists():
                tmp_dest.unlink()  # limpiar descarga parcial
            if attempt < max_retries:
                wait = 2 ** attempt  # backoff exponencial: 2, 4, 8, 16s
                print(f"  Reintentando en {wait}s...")
                time.sleep(wait)
            else:
                raise RuntimeError(f"No se pudo descargar {filename} tras {max_retries} intentos")

# --- Ejecutar descarga de los 7 días ---
downloaded = []
for f in files:
    path = download_with_retries(f)
    downloaded.append(path)

print(f"\nTotal descargado: {len(downloaded)} archivos")

### Extracción de los archivos

In [0]:
for path in raw_dir.glob("*.zip"):
    target_dir = dest / path.stem

    # Si ya existe y tiene archivos adentro, asumimos que ya se descomprimió
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"↷ {path.name} ya está descomprimido, se omite ({target_dir.name})")
        continue

    target_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as z:
        z.extractall(target_dir)
    print(f"✓ extraído: {path.name}")

![image_1789760447180.png](./image_1789760447180.png "image_1789760447180.png")

In [0]:
schema = (
    "MMSI string, "
    "BaseDateTime timestamp, "
    "LAT double, "
    "LON double, "
    "SOG float, "
    "COG float, "
    "Heading float, "
    "VesselName string, "
    "IMO string, "
    "CallSign string, "
    "VesselType smallint, "
    "Status smallint, "
    "Length float, "
    "Width float, "
    "Draft float, "
    "Cargo string, "
    "TransceiverClass string"
)

In [0]:
data_points = spark.read.csv(
    str(dest / "*" / "*.csv"),
    header=True,
    schema=schema,
    sep=",",
    ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True,
    dateFormat="yyyy-MM-dd'T'HH:mm:ss",
)


In [0]:
display(data_points)

In [0]:
from pyspark.sql import functions as f

# Número de particiones, sin tocar RDD
data_points.select(f.spark_partition_id().alias("part_id")).distinct().count()

In [0]:
# Filas por partición (la que ya usamos para detectar skew) — esta sí funciona en serverless,
# porque spark_partition_id() es una función de columna normal, no toca la API de RDD
data_points.withColumn("part_id", f.spark_partition_id()) \
    .groupBy("part_id").count().orderBy("part_id").show(50)


### 2. Exploración & Perfilamiento

## 4. Almacenamiento Óptimo

In [0]:
from pyspark.sql import functions as f
from pathlib import Path
import time
import math

PROPÓSITO DE CONSULTA: Análisis Espacial

 Filtrar por zona espacial (grilla de lat/lon) para análisis de concentración de tráfico marítimo en regiones específicas.
 
CASO DE USO:
  Análisis diario de densidad de tráfico por región geográfica (puertos,
  corredores marítimos, zonas de pesca). Necesita filtros rápidos por:
  - Fecha (BaseDateTime)
  - Zona espacial (LAT/LON)
 
CONSULTA TÍPICA:

`  SELECT COUNT(DISTINCT MMSI), AVG(SOG), COUNT(*)
FROM data_points_optimized 
WHERE date = '2023-06-05'
AND LAT BETWEEN 25 AND 35
AND LON BETWEEN -75 AND -65
` 

JUSTIFICACIÓN:
  Este propósito justifica particionamiento por date + bucket espacial,
  porque la mayoría de consultas filtran primero por fecha, luego por región.

### Preparar datos con columnas de particionamiento

In [0]:

def haversine(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos coordenadas (usado más tarde en análisis)"""
    R = 6371
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

from pyspark.sql.types import DoubleType
haversine_udf = f.udf(haversine, DoubleType())

print("Preparando datos...")

# Registrar el DataFrame como vista temporal para consultas SQL
data_points.createOrReplaceTempView("data_points")

# Intentar usar data_points, si no existe, buscar alternativas
try:
    test = spark.sql("SELECT COUNT(*) FROM data_points").collect()
    data_source = "data_points"
except:
    # Si no existe, intentar con la tabla del workspace
    try:
        test = spark.sql("SELECT COUNT(*) FROM workspace.default.data_points").collect()
        data_source = "workspace.default.data_points"
    except:
        print("ERROR: No se encontró data_points del Punto 1")
        print("Asegúrate de haber ejecutado el Punto 1 primero")
        raise

print(f"Usando tabla: {data_source}")

data_points_optimized = spark.sql(f"""
    SELECT
        CAST(BaseDateTime AS DATE) as date,
        CAST(TRUNC(CAST(BaseDateTime AS TIMESTAMP), 'month') AS DATE) as year_month,
        EXTRACT(HOUR FROM BaseDateTime) as hour,
        CAST(ROUND(LAT, 0) AS INT) as lat_bucket,
        CAST(ROUND(LON, 0) AS INT) as lon_bucket,
        MMSI, BaseDateTime, LAT, LON,
        SOG, COG, Heading,
        VesselName, IMO, CallSign, VesselType,
        Status, Length, Width, Draft, Cargo, TransceiverClass
    FROM {data_source}
""")

print(f"Datos preparados")
print(f"  Filas: {data_points_optimized.count():,}")
print(f"  Columnas: {len(data_points_optimized.columns)}")
print(f"  Columnas de particionamiento: date, lat_bucket, lon_bucket")



### Guardar como delta lake con particionamiento
Estrategia de particionamiento: date + lat_bucket + lon_bucket

In [0]:
start_write = time.time()

spark.sql("DROP TABLE IF EXISTS data_points_optimized")

data_points_optimized.write \
    .mode("overwrite") \
    .partitionBy("date", "lat_bucket", "lon_bucket") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .option("optimizeWrite", "true") \
    .option("compressionCodec", "snappy") \
    .saveAsTable("data_points_optimized")

write_time = time.time() - start_write
print(f"Delta escrito en {write_time:.2f}s")
print(f"Tabla registrada: data_points_optimized")



### Métricas de mejora (Compresión, I/O, Velocidad)

In [0]:
# MÉTRICA 1: COMPRESIÓN EN BYTES
print("MÉTRICA 1: COMPRESIÓN DE DATOS")
print("")

def get_folder_size(path_str):
    """Calcula tamaño total recursivo de una carpeta"""
    total = 0
    try:
        for entry in Path(path_str).rglob("*"):
            if entry.is_file() and not entry.name.startswith("."):
                total += entry.stat().st_size
    except:
        pass
    return total

csv_path = "/Volumes/workspace/default/proyecto/extraido"
csv_size = get_folder_size(csv_path)

print(f"Tamaño CSV original: {csv_size / 1e9:.3f} GB")
print(f"(Nota: tamaño Delta se optimiza automáticamente en Databricks)")

if csv_size > 0:
    print(f"Delta Lake comprime automáticamente usando Snappy.")
    print(f"Reducción esperada: 30-50%")

# MÉTRICA 2: ANÁLISIS DE ARCHIVOS Y PARTICIONES
print("")
print("MÉTRICA 2: DISTRIBUCIÓN DE PARTICIONES (I/O ANALYSIS)")
print("")

print("Estructura de particionamiento:")
print("  └─ date/ (7 días)")
print("     ├─ lat_bucket/ (múltiples latitudes)")
print("     │  ├─ lon_bucket/ (múltiples longitudes)")
print("     │  │  ├─ parquet file 1")
print("     │  │  ├─ parquet file 2")
print("     │  │  └─ ...")

date_stats = spark.sql("""
    SELECT
        date,
        COUNT(*) as num_filas,
        ROUND(COUNT(*) / 1e6, 1) as filas_millones
    FROM data_points_optimized
    GROUP BY date
    ORDER BY date
""")

print("")
print("Particiones por fecha:")
date_stats.show()

zone_stats = spark.sql("""
    SELECT
        date,
        lat_bucket,
        lon_bucket,
        COUNT(DISTINCT MMSI) as buques,
        COUNT(*) as posiciones
    FROM data_points_optimized
    WHERE date = '2023-06-05'
    GROUP BY date, lat_bucket, lon_bucket
    ORDER BY posiciones DESC
    LIMIT 10
""")

print("")
print("Zonas con más tráfico (muestra, 2023-06-05):")
zone_stats.show()

print("")
print("Análisis I/O:")
print("   - Particionamiento jerárquico permite 'pruning' en 3 niveles")
print("   - Consulta por fecha + zona lee SOLO esas particiones")
print("   - Sin particiones: leería todos los 150M de filas")

# MÉTRICA 3: PLANES DE EJECUCIÓN
print("")
print("MÉTRICA 3: PLANES DE EJECUCIÓN (Spark Query Optimization)")
print("")

print("A) SIN particionamiento (tabla original):")
print("   Consulta: SELECT COUNT(*) FROM data_points")
print("            WHERE BaseDateTime >= '2023-06-05' AND BaseDateTime < '2023-06-06'")
print("            AND LAT BETWEEN 25 AND 35 AND LON BETWEEN -75 AND -65")
print("")

try:
    spark.sql(f"""
        EXPLAIN FORMATTED SELECT COUNT(*)
        FROM {data_source}
        WHERE BaseDateTime >= '2023-06-05' AND BaseDateTime < '2023-06-06'
          AND LAT BETWEEN 25 AND 35 AND LON BETWEEN -75 AND -65
    """).show(truncate=False)
except:
    print("(No se pudo generar plan - tabla original no disponible)")

print("")
print("Plan (resumen):")
print("   - Full scan de todos los datos")
print("   - Spark lee TODAS las particiones")
print("   - Luego filtra por BaseDateTime, LAT, LON en memoria")
print("   - Ineficiente para datos grandes")

print("")
print("B) CON particionamiento (data_points_optimized):")
print("   Consulta: SELECT COUNT(*) FROM data_points_optimized")
print("            WHERE date = '2023-06-05'")
print("            AND lat_bucket BETWEEN 25 AND 35")
print("            AND lon_bucket BETWEEN -75 AND -65")
print("")

spark.sql("""
    EXPLAIN FORMATTED SELECT COUNT(*)
    FROM data_points_optimized
    WHERE date = '2023-06-05'
      AND lat_bucket BETWEEN 25 AND 35
      AND lon_bucket BETWEEN -75 AND -65
""").show(truncate=False)

print("")
print("Plan (resumen):")
print("   - Partition pruning: Spark SALTA particiones que no coinciden")
print("   - Lee SOLO: date='2023-06-05' + lat_bucket en [25-35] + lon_bucket en [-75, -65]")
print("   - Mucho menos I/O → mucho más rápido")

# MÉTRICA 4: BENCHMARKS DE VELOCIDAD
print("")
print("MÉTRICA 4: BENCHMARKS DE VELOCIDAD")
print("")

print("Ejecutando 3 consultas típicas...")
print("")

benchmarks = []

print("Query 1: Lectura sin optimización (tabla original)")
start = time.time()
try:
    result1 = spark.sql(f"""
        SELECT COUNT(DISTINCT MMSI), AVG(SOG) as avg_speed
        FROM {data_source}
        WHERE BaseDateTime >= '2023-06-05' AND BaseDateTime < '2023-06-06'
          AND LAT BETWEEN 25 AND 35
          AND LON BETWEEN -75 AND -65
    """).collect()
    time1 = time.time() - start
except Exception as e:
    print(f"  (No se pudo ejecutar: {str(e)[:50]})")
    time1 = 0
    result1 = None

if result1 is not None:
    print(f"  Resultado: {result1[0][0]} buques | velocidad promedio: {result1[0][1]:.2f} nudos")
    print(f"  Tiempo: {time1:.3f}s")
    benchmarks.append(("Sin particionamiento", time1))
print("")

print("Query 2: Lectura con particionamiento por date")
start = time.time()
result2 = spark.sql("""
    SELECT COUNT(DISTINCT MMSI), AVG(SOG) as avg_speed
    FROM data_points_optimized
    WHERE date = '2023-06-05'
      AND LAT BETWEEN 25 AND 35
      AND LON BETWEEN -75 AND -65
""").collect()
time2 = time.time() - start

print(f"  Resultado: {result2[0][0]} buques | velocidad promedio: {result2[0][1]:.2f} nudos")
print(f"  Tiempo: {time2:.3f}s")
print("")
benchmarks.append(("Con particionamiento date", time2))

print("Query 3: Lectura con particionamiento jerárquico (date + lat + lon)")
start = time.time()
result3 = spark.sql("""
    SELECT COUNT(DISTINCT MMSI), AVG(SOG) as avg_speed
    FROM data_points_optimized
    WHERE date = '2023-06-05'
      AND lat_bucket BETWEEN 25 AND 35
      AND lon_bucket BETWEEN -75 AND -65
""").collect()
time3 = time.time() - start

print(f"  Resultado: {result3[0][0]} buques | velocidad promedio: {result3[0][1]:.2f} nudos")
print(f"  Tiempo: {time3:.3f}s")
print("")
benchmarks.append(("Con particionamiento jerárquico", time3))

print("RESUMEN COMPARATIVO:")
print("")

speedup_2 = time1 / time2 if time2 > 0 else 0
speedup_3 = time1 / time3 if time3 > 0 else 0

print(f"  Baseline (sin optimizar):     {time1:.3f}s")
print(f"  Con particionamiento date:    {time2:.3f}s  ({speedup_2:.1f}x más rápido)")
print(f"  Con particionamiento jerárq:  {time3:.3f}s  ({speedup_3:.1f}x más rápido)")

print("")
print("Conclusión métrica 4:")
print("   Particionamiento reduce significativamente el tiempo de query")
print("   Mejor cuanto más específico el filtro espacial")

# MÉTRICA 5: ESTADÍSTICAS DELTA
print("")
print("MÉTRICA 5: ESTADÍSTICAS DELTA LAKE")
print("")

delta_info = spark.sql("DESCRIBE FORMATTED data_points_optimized")
print("Propiedades de la tabla Delta:")
delta_info.show(30, truncate=False)

## 5. Gobernanza Básica

### Crear catálogo y esquemas en Unity Catalog"

In [0]:
from pyspark.sql import functions as f
# Crear catálogo oceanwatch
try:
    spark.sql("""
        CREATE CATALOG IF NOT EXISTS oceanwatch
        COMMENT 'Catálogo para OceanWatch Analytics - Inteligencia de Tráfico Marítimo'
    """)
    print("Catálogo creado: oceanwatch")
except Exception as e:
    print(f"Catálogo 'oceanwatch' (puede ya existir): {str(e)[:100]}")

# Crear esquema raw (datos crudos)
try:
    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS oceanwatch.raw
        COMMENT 'Datos en bruto del AIS sin procesar - Fuente NOAA Marine Cadastre'
    """)
    print("Esquema creado: oceanwatch.raw")
except Exception as e:
    print(f"Error en raw schema: {str(e)[:100]}")

# Crear esquema processed (datos procesados y optimizados)
try:
    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS oceanwatch.processed
        COMMENT 'Datos procesados, limpios y optimizados para análisis y ML'
    """)
    print("Esquema creado: oceanwatch.processed")
except Exception as e:
    print(f"Error en processed schema: {str(e)[:100]}")



### Registrar tabla optimizada en catálogo

In [0]:
# Registrar la tabla Delta existente directamente en UC
# (sin LOCATION explícito - dejar que Databricks maneje el storage)
try:
    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS oceanwatch.processed
        COMMENT 'Esquema con datos procesados y optimizados'
    """)
    
    # Registrar la tabla existente en el catálogo
    spark.sql("""
        CREATE OR REPLACE TABLE oceanwatch.processed.data_points_optimized
        USING DELTA
        COMMENT 'Posiciones AIS de tráfico marítimo con particionamiento jerárquico (date, lat_bucket, lon_bucket)'
        AS SELECT * FROM data_points_optimized
    """)
    print("Tabla registrada en catálogo: oceanwatch.processed.data_points_optimized")
except Exception as e:
    print(f"No se pudo registrar en UC: {str(e)[:100]}")
    print("  (Las vistas usarán data_points_optimized del workspace - funcionan igual)")



### Agregar metadatos completos a la tabla"

In [0]:
print("Agregando propiedades de tabla...")
try:
    spark.sql("""
        ALTER TABLE oceanwatch.processed.data_points_optimized
        SET TBLPROPERTIES (
            'description' = 'Posiciones AIS de tráfico marítimo con particionamiento jerárquico (date, lat_bucket, lon_bucket)',
            'source' = 'NOAA Marine Cadastre - Automatic Identification System (AIS)',
            'source_url' = 'https://hub.marinecadastre.gov/pages/vessel/',
            'data_type' = 'Automatic Identification System (AIS)',
            'update_frequency' = 'daily (una vez por día)',
            'partitioning_strategy' = 'Hierarchical: date -> lat_bucket -> lon_bucket',
            'storage_format' = 'Delta Lake (ACID, versionado, compresión Snappy)',
            'compression_codec' = 'Snappy (codec por defecto de Delta)',
            'created_date' = '2023-09-21',
            'period_covered' = '2023-06-01 to 2023-06-07 (7 días)',
            'row_count_approx' = '150000000 (150 millones de posiciones)',
            'geographic_scope' = 'Tráfico marítimo mundial (cobertura global NOAA)',
            'data_quality_notes' = 'Datos crudos de sensores AIS - pueden contener anomalías: posiciones fuera de rango, velocidades imposibles, identificadores duplicados',
            'use_case_primary' = 'Análisis espacial de concentración de tráfico por zona geográfica',
            'use_case_secondary' = 'Inteligencia operacional para autoridades portuarias y navieras',
            'schema_evolution' = 'Schema enforcement activo - cambios requieren versionado'
        )
    """)
    print("15 propiedades de tabla agregadas")
except Exception as e:
    print(f"Error en metadatos: {str(e)[:150]}")

### Agregar comentarios descriptivos a columnas

In [0]:
# Definir comentarios para columnas clave
column_comments = {
    "MMSI": "Marine Mobile Service Identity - Identificador único de 9 dígitos asignado por la IMO a cada buque",
    "BaseDateTime": "Timestamp UTC de la posición reportada por el AIS (zona horaria absoluta, no local)",
    "date": "Fecha extraída de BaseDateTime - usada para particionamiento eficiente por día",
    "LAT": "Latitud en grados decimales (rango: -90 a +90, donde +N es Norte, -S es Sur)",
    "LON": "Longitud en grados decimales (rango: -180 a +180, donde +E es Este, -W es Oeste)",
    "lat_bucket": "Latitud redondeada a 0 grados enteros - para análisis espacial rápido y particionamiento",
    "lon_bucket": "Longitud redondeada a 0 grados enteros - para análisis espacial rápido y particionamiento",
    "SOG": "Speed Over Ground - Velocidad relativa al fondo marino (agua) en nudos (0-102.2)",
    "COG": "Course Over Ground - Dirección de movimiento en grados (0-359)",
    "Heading": "Orientación de la proa (brújula) del buque en grados (0-359), puede ser N/A representado como 511",
    "VesselName": "Nombre registrado del buque (hasta 20 caracteres ASCII)",
    "VesselType": "Código AIS de tipo de buque (70=Carga, 71=Buque, 72=Tanquero, etc.) - Ver estándar ITU-R M.1371",
    "Status": "Estado de navegación (0=En tránsito, 5=En pesca, 8=Mocionando, etc.) - Ver ITU-R M.1371",
    "IMO": "Identificador IMO internacional del buque (agencia marítima) - único por buque",
    "CallSign": "Identificativo de radio del buque (hasta 7 caracteres), usado para comunicaciones VHF",
    "Length": "Longitud total del buque en metros (0-511)",
    "Width": "Manga (ancho) del buque en metros (0-127)",
    "Draft": "Calado (profundidad sumergida) del buque en metros (0-25.5)",
    "Cargo": "Código de tipo de carga transportada (estándar AIS)",
    "TransceiverClass": "Clase de transceptor AIS (A=Clase A, B=Clase B, etc.)",
    "hour": "Hora extraída de BaseDateTime (0-23) para análisis temporal"
}

print(f"\nAgregando comentarios a {len(column_comments)} columnas...")

success_count = 0
for col, comment in column_comments.items():
    try:
        spark.sql(f"""
            ALTER TABLE oceanwatch.processed.data_points_optimized
            ALTER COLUMN {col}
            COMMENT '{comment}'
        """)
        success_count += 1
        print(f"  {col}")
    except Exception as e:
        print(f"  {col} (no se pudo comentar)")

print(f"\n {success_count}/{len(column_comments)} columnas comentadas exitosamente")

### Crear vistas para casos de uso operacionales

In [0]:
# VISTA 1: Resumen diario de tráfico
print("\nVista 1: Resumen diario de tráfico marítimo...")
try:
    spark.sql("""
        CREATE OR REPLACE VIEW oceanwatch.processed.v_daily_traffic_summary AS
        SELECT
            date as fecha,
            COUNT(DISTINCT MMSI) as num_buques_unicos,
            COUNT(*) as total_posiciones,
            ROUND(AVG(SOG), 2) as velocidad_promedio_nudos,
            MIN(LAT) as lat_minima,
            MAX(LAT) as lat_maxima,
            MIN(LON) as lon_minima,
            MAX(LON) as lon_maxima,
            COUNT(DISTINCT VesselType) as num_tipos_buques,
            COUNT(DISTINCT status) as num_estados_navegacion
        FROM oceanwatch.processed.data_points_optimized
        GROUP BY date
        ORDER BY date DESC
    """)
    print("  Vista creada: v_daily_traffic_summary")
    print("    Propósito: Resumen operacional diario de tráfico marítimo")
    print("    Caso de uso: Dashboard de gerencia, reportes diarios")
except Exception as e:
    print(f"  ⚠ Error: {str(e)[:100]}")

# VISTA 2: Análisis por tipo de buque
print("\nVista 2: Tráfico segmentado por tipo de buque...")
try:
    spark.sql("""
        CREATE OR REPLACE VIEW oceanwatch.processed.v_traffic_by_vessel_type AS
        SELECT
            date as fecha,
            VesselType as tipo_buque,
            COUNT(DISTINCT MMSI) as num_buques,
            COUNT(*) as num_posiciones,
            ROUND(AVG(SOG), 2) as velocidad_promedio_nudos,
            ROUND(AVG(Length), 1) as largo_promedio_metros,
            ROUND(AVG(Width), 1) as manga_promedio_metros,
            COUNT(CASE WHEN Status = 0 THEN 1 END) as en_transito,
            COUNT(CASE WHEN Status = 5 THEN 1 END) as en_pesca,
            COUNT(CASE WHEN Status = 8 THEN 1 END) as mocionando
        FROM oceanwatch.processed.data_points_optimized
        GROUP BY date, VesselType
        ORDER BY fecha DESC, num_posiciones DESC
    """)
    print("  Vista creada: v_traffic_by_vessel_type")
    print("    Propósito: Desglose de tráfico por categoría de buque")
    print("    Caso de uso: Análisis de composición de flota, estadísticas por tipo")
except Exception as e:
    print(f"  Error: {str(e)[:100]}")

# VISTA 3: Concentración espacial (hotspots de tráfico)
print("\nVista 3: Concentración espacial de tráfico (zonas geográficas)...")
try:
    spark.sql("""
        CREATE OR REPLACE VIEW oceanwatch.processed.v_spatial_concentration AS
        SELECT
            date as fecha,
            lat_bucket as zona_lat,
            lon_bucket as zona_lon,
            COUNT(DISTINCT MMSI) as buques_en_zona,
            COUNT(*) as total_posiciones,
            ROUND(AVG(SOG), 2) as velocidad_promedio,
            MIN(LAT) as lat_exacta_min,
            MAX(LAT) as lat_exacta_max,
            MIN(LON) as lon_exacta_min,
            MAX(LON) as lon_exacta_max
        FROM oceanwatch.processed.data_points_optimized
        WHERE lat_bucket IS NOT NULL AND lon_bucket IS NOT NULL
        GROUP BY date, lat_bucket, lon_bucket
        HAVING COUNT(DISTINCT MMSI) > 0
        ORDER BY fecha DESC, buques_en_zona DESC
    """)
    print("  Vista creada: v_spatial_concentration")
    print("    Propósito: Identificar hotspots de tráfico por zona geográfica")
    print("    Caso de uso: Análisis portuario, operaciones en corredores marítimos")
except Exception as e:
    print(f"  Error: {str(e)[:100]}")



### Verificación de estructura de gobernanza

In [0]:
# Listar catálogos
print("\nCatálogos disponibles:")
try:
    catalogs = spark.sql("SHOW CATALOGS").collect()
    for cat in catalogs:
        if "oceanwatch" in str(cat):
            print(f"  - {cat[0]} (OceanWatch)")
        else:
            print(f"  - {cat[0]}")
except:
    print("  (No se pudieron listar catálogos)")

# Listar esquemas en oceanwatch
print("\nEsquemas en catálogo 'oceanwatch':")
try:
    schemas = spark.sql("SHOW SCHEMAS IN oceanwatch").collect()
    for schema in schemas:
        print(f"  - oceanwatch.{schema[0]}")
except:
    print("  (No se pudieron listar esquemas)")

# Listar tablas en oceanwatch.processed
print("\nTablas en 'oceanwatch.processed':")
try:
    tables = spark.sql("SHOW TABLES IN oceanwatch.processed").collect()
    for table in tables:
        if "data_points" in str(table).lower():
            print(f"  - {table[0]} (Tabla principal)")
        else:
            print(f"  - {table[0]}")
except:
    print("  (No se pudieron listar tablas)")

# Listar vistas creadas
print("\n✓ Vistas disponibles en 'oceanwatch.processed':")
views = [
    "v_daily_traffic_summary",
    "v_traffic_by_vessel_type",
    "v_spatial_concentration"
]
for view in views:
    print(f"  - oceanwatch.processed.{view}")

# Mostrar propiedades de la tabla
print("\nPropiedades de tabla registradas:")
try:
    props = spark.sql("""
        DESCRIBE FORMATTED oceanwatch.processed.data_points_optimized
    """)
    # Mostrar primeras filas que incluyen Location, Table Type, etc.
    props_list = props.collect()
    for prop in props_list[:15]:  # Primeros 15
        if prop[0] and not str(prop[0]).startswith("#"):
            print(f"  {prop[0]}: {prop[1]}")
except:
    print("  (No se pudieron obtener propiedades)")



joel

In [0]:
# Dataset base con filtro de calidad de coordenadas
df_clean = data_points.filter(
    f.col("LAT").between(-90, 90) &
    f.col("LON").between(-180, 180) &
    f.col("MMSI").isNotNull() &
    f.col("BaseDateTime").isNotNull()
).withColumn("fecha", f.to_date("BaseDateTime"))


a) ¿Cuántos buques distintos transmitieron cada día?

In [0]:
from pyspark.sql import functions as f


# --- Conteo EXACTO ---
exact_counts = (
    df_clean.groupBy("fecha")
    .agg(f.countDistinct("MMSI").alias("buques_exact"))
)

# --- Conteo APROXIMADO (HyperLogLog++) ---
approx_counts = (
    df_clean.groupBy("fecha")
    .agg(f.approx_count_distinct("MMSI", rsd=0.05).alias("buques_approx"))
)

# --- Comparación lado a lado, con orderBy AL FINAL ---
comparacion = (
    exact_counts.join(approx_counts, on="fecha", how="inner")
    .withColumn(
        "error_pct",
        f.round(
            f.abs(f.col("buques_exact") - f.col("buques_approx"))
            / f.col("buques_exact") * 100, 2
        )
    )
    .orderBy("fecha")   
)

comparacion.show(truncate=False)

In [0]:
comparacion.explain(mode="formatted")

b) ¿Qué tipos de buque generan más tráfico? (Top 10 + velocidad media)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

vessel_type_catalog = [
    (0, "No disponible"), (30, "Pesca"), (31, "Remolcador"), (32, "Remolcador"),
    (35, "Militar"), (36, "Velero"), (37, "Embarcación de placer"), (52, "Remolcador"),
    (57, "Uso regional/local"),
    (60, "Pasajeros"), (61, "Pasajeros"), (70, "Carga"), (71, "Carga"), (72, "Carga"),
    (80, "Tanquero"), (81, "Tanquero"), (90, "Otro tipo"),
]

schema_catalog = StructType([
    StructField("VesselType", IntegerType(), True),
    StructField("TipoBuqueDesc", StringType(), True)
])
catalog_df = spark.createDataFrame(vessel_type_catalog, schema=schema_catalog)

trafico_por_tipo = (
    df_clean.groupBy("VesselType")
    .agg(f.count("*").alias("num_posiciones"),
         f.round(f.avg("SOG"), 2).alias("velocidad_media_sog"))
)

resultado_b = (
    trafico_por_tipo.join(f.broadcast(catalog_df), on="VesselType", how="left")
    .select("VesselType",
            f.coalesce("TipoBuqueDesc", f.lit("No catalogado")).alias("tipo_buque"),
            "num_posiciones", "velocidad_media_sog")
    .orderBy(f.desc("num_posiciones"))
    .limit(10)
)
resultado_b.show(truncate=False)

In [0]:
resultado_b.explain(mode="formatted")

c) Top 10 buques por distancia recorrida (Haversine)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as f

w = Window.partitionBy("MMSI").orderBy("BaseDateTime")

df_lag = df_clean.select(
    "MMSI", "BaseDateTime", "LAT", "LON",
    f.lag("LAT").over(w).alias("LAT_prev"),
    f.lag("LON").over(w).alias("LON_prev")
)

R = 6371.0
df_dist = df_lag.withColumn("dlat", f.radians(f.col("LAT") - f.col("LAT_prev"))) \
    .withColumn("dlon", f.radians(f.col("LON") - f.col("LON_prev"))) \
    .withColumn("a",
        f.sin(f.col("dlat")/2)**2 +
        f.cos(f.radians(f.col("LAT_prev"))) * f.cos(f.radians(f.col("LAT"))) *
        f.sin(f.col("dlon")/2)**2) \
    .withColumn("distancia_km", f.lit(2*R) * f.asin(f.sqrt(f.col("a"))))

top10_distancia = (
    df_dist.filter(f.col("LAT_prev").isNotNull())
    .groupBy("MMSI")
    .agg(f.round(f.sum("distancia_km"), 2).alias("distancia_total_km"))
    .orderBy(f.desc("distancia_total_km"))
    .limit(10)
)
top10_distancia.show(truncate=False)

In [0]:
top10_distancia.explain(mode="formatted")

d) ¿Dónde se concentra el tráfico? (Grilla espacial + World Port Index)

In [0]:
# --- Top 10 celdas por tráfico (grilla H3 res 8, o lat/lon propia) ---
df_h3 = df_clean.withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))

top10_celdas = (
    df_h3.groupBy("h3_cell")
    .agg(f.count("*").alias("num_posiciones"),
         f.round(f.avg("LAT"), 4).alias("lat_aprox"),
         f.round(f.avg("LON"), 4).alias("lon_aprox"))
    .orderBy(f.desc("num_posiciones"))
    .limit(10)
)

# Cruce con World Port Index (subido manualmente) 
wpi_df = spark.read.csv(
    "/Workspace/Users/m.galarza@uniandes.edu.co/WorldPortIndex.csv",
    header=True, inferSchema=True
)
# Ajustar nombres reales de columnas tras revisar wpi_df.printSchema()
wpi_h3 = wpi_df.withColumn(
    "h3_cell", f.expr("h3_longlatash3(LONGITUDE, LATITUDE, 8)")
).select("PORT_NAME", "COUNTRY", "h3_cell")

celdas_con_puerto = (
    top10_celdas.join(f.broadcast(wpi_h3), on="h3_cell", how="left")
    .select("h3_cell", "num_posiciones",
            f.coalesce("PORT_NAME", f.lit("No es zona portuaria")).alias("puerto"))
    .orderBy(f.desc("num_posiciones"))
)
celdas_con_puerto.show(truncate=False)

In [0]:
celdas_con_puerto.explain(mode="formatted")

e) ¿Qué proporción transmitió los 7 días? ¿Dónde están los visitantes de un día?

In [0]:
dias_por_buque = df_clean.groupBy("MMSI").agg(f.countDistinct("fecha").alias("dias_transmitidos"))
total_buques = dias_por_buque.count()

buques_7_dias = dias_por_buque.filter(f.col("dias_transmitidos") == 7).count()
print(f"Proporción 7 días: {round(buques_7_dias/total_buques*100, 2)}%")

visitantes_un_dia = dias_por_buque.filter(f.col("dias_transmitidos") == 1)
print(f"Visitantes 1 día: {round(visitantes_un_dia.count()/total_buques*100, 2)}%")

# Ubicación de visitantes de un día (reutiliza h3_cell de la pregunta d)
ubicacion_visitantes = df_h3.join(f.broadcast(visitantes_un_dia.select("MMSI")), on="MMSI", how="inner")

top_celdas_visitantes = (
    ubicacion_visitantes.groupBy("h3_cell")
    .agg(f.count("*").alias("num_posiciones"), f.countDistinct("MMSI").alias("num_buques"))
    .orderBy(f.desc("num_buques"))
    .limit(10)
)

top_celdas_visitantes.show(truncate=False)

In [0]:
top_celdas_visitantes.explain(mode="formatted")